# VERA on Kaggle — UCF-Crime Inference + AUC

This notebook runs the VERA pipeline on Kaggle GPUs.

**Two paths:**
- **Path A (fast, no GPU):** use pre-computed segment scores + vision features from the paper's Google Drive links, then compute AUC.
- **Path B (GPU):** run InternVL2-8B on extracted frames to generate scores yourself.

**Before running:**
1. Enable GPU: Settings -> Accelerator -> GPU T4 x2 (only needed for Path B)
2. Enable Internet: Settings -> Internet -> On
3. Upload the VERA repo as a Kaggle Dataset (or clone from GitHub in the next cell).

## 0. Setup — copy code to a writable location
`/kaggle/input` is read-only. VERA writes output files, so we copy it to `/kaggle/working`.

In [ ]:
import os, shutil

# EDIT THIS to match your uploaded dataset path, e.g. /kaggle/input/vera-code/VERA
SRC = '/kaggle/input/vera-code/VERA'
DST = '/kaggle/working/VERA'

if os.path.exists(SRC):
    if os.path.exists(DST):
        shutil.rmtree(DST)
    shutil.copytree(SRC, DST)
    print('Copied repo to', DST)
else:
    print('SRC not found. Either upload the repo as a dataset or clone from GitHub in the next cell.')

os.chdir(DST if os.path.exists(DST) else '/kaggle/working')
print('CWD:', os.getcwd())

In [ ]:
# OPTIONAL: clone from GitHub instead of uploading a dataset
# !git clone https://github.com/YOUR_USERNAME/VERA.git /kaggle/working/VERA
# import os; os.chdir('/kaggle/working/VERA'); print(os.getcwd())

## 1. Install dependencies

In [ ]:
!pip install -q scikit-learn scipy numpy
# Only needed for Path B (running the model):
# !pip install -q transformers pytorch-lightning decord timm einops sentencepiece peft accelerate bitsandbytes

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

---
# PATH A — Fast reproduce (no GPU)

Download the paper's pre-computed **segment scores** and **vision features**, then run `compute_auc.py`.

`compute_auc.py` expects:
- `Data/UCF_Eval.json` (already in repo)
- `Data/vision_features/<video>.npy` (ImageBind features)
- `Data/segment_level_score/<video>.json` (segment scores)

Because the Google Drive files are large, the easiest route is to **download them once, upload as a Kaggle dataset**, then point the paths below to them.

In [ ]:
import os, shutil

# EDIT these to where you uploaded the features/scores as Kaggle datasets
VISION_SRC = '/kaggle/input/vera-ucf-vision-features'   # folder of *.npy
SCORE_SRC  = '/kaggle/input/vera-ucf-segment-scores'    # folder of *.json

os.makedirs('Data/vision_features', exist_ok=True)
os.makedirs('Data/segment_level_score', exist_ok=True)

def link_all(src, dst):
    if not os.path.isdir(src):
        print('missing:', src); return
    n = 0
    for f in os.listdir(src):
        s = os.path.join(src, f)
        d = os.path.join(dst, f)
        if not os.path.exists(d):
            shutil.copy(s, d); n += 1
    print(f'copied {n} files into {dst}')

link_all(VISION_SRC, 'Data/vision_features')
link_all(SCORE_SRC, 'Data/segment_level_score')

In [ ]:
# compute_auc.py at repo root reads from Data/. Run it directly.
!python compute_auc.py

The last printed number is the **frame-level ROC-AUC** for UCF-Crime.

> Note: the root `compute_auc.py` also reads from `./scores_77/tests/`. If you only have the `segment_level_score` folder, use the version under `Inference/` or comment out the `scores_77` glob block. See the fix cell below if you hit a `FileNotFoundError`.

In [ ]:
# If compute_auc.py errors on the './scores_77/tests/' glob, make an empty dir so the glob returns nothing:
import os
os.makedirs('scores_77/tests', exist_ok=True)
print('created empty scores_77/tests to satisfy the glob')

---
# PATH B — Generate scores with InternVL2-8B (needs GPU)

This runs the model on extracted frames. Requirements:
- Frames uploaded to `Data/ucf/frames/<Category>/<VideoName>/000001.jpg ...`
- `Data/UCF_Eval.json` (in repo)
- ~16GB VRAM (T4). We disable flash-attn and enable 8-bit to fit.

**Storage warning:** full UCF-Crime frames are 50GB+ and will NOT fit in Kaggle. Use a small subset (10-20 videos).

In [ ]:
# Install model deps (only for Path B)
!pip install -q transformers pytorch-lightning decord timm einops sentencepiece peft accelerate bitsandbytes

In [ ]:
# Patch generate_initial_score.py for Kaggle T4:
#  - disable flash attention (not supported on T4)
#  - enable 8-bit loading to fit in 16GB
#  - point vis_root to your uploaded frames
import re

path = 'generate_initial_score.py'
with open(path, 'r', encoding='utf-8') as f:
    src = f.read()

src = src.replace('use_flash_attn=True', 'use_flash_attn=False')
# add load_in_8bit right after the low_cpu_mem_usage line (idempotent-ish)
if 'load_in_8bit=True' not in src:
    src = src.replace('low_cpu_mem_usage=True,', 'low_cpu_mem_usage=True,\n    load_in_8bit=True,')

# EDIT this to your uploaded frames path if different
src = src.replace("vis_root='Data/ucf/frames/'", "vis_root='/kaggle/input/ucf-frames-subset/frames/'")

with open(path, 'w', encoding='utf-8') as f:
    f.write(src)
print('patched', path)

In [ ]:
# Run inference. First run downloads InternVL2-8B (~16GB) to ./cache.
!python generate_initial_score.py

In [ ]:
# Scores are written to prediction_scores_InternVL2/<video>.json
import os, glob
outs = glob.glob('prediction_scores_InternVL2/*.json')
print(len(outs), 'score files generated')
print(outs[:5])

## Notes
- For AUC after Path B, you still need the **vision features** (`Data/vision_features/*.npy`) which come from ImageBind. Generate them with LAVAD's feature extractor or download the paper's pre-computed features.
- Kaggle sessions cap at ~9-12h and 30h/week; the full dataset won't finish. Stick to a subset.
- Everything under `/kaggle/working` persists as notebook output when you commit.